# SHL Research Engineer Assessment: Spoken Grammar Scoring Engine
**Author:** Candidate (Research Engineer Applicant)  
**Competition:** Private Kaggle Challenge - Grammar Scoring for Spoken Data  
**Primary Evaluation Metrics:** Pearson Correlation Coefficient ($r$) & Root Mean Squared Error ($\text{RMSE}$)  
**Mandatory Submission Requirement:** Display **Training Data RMSE** in the notebook.

---

## 1. Executive Summary & Problem Formulation

### 1.1 Objective
The goal is to develop an automated continuous scoring engine that evaluates spoken English responses (45–60 seconds long) on a continuous **Mean Opinion Score (MOS) scale from 0.0 to 5.0**, aligning with the official human rubric:
* **Score 1:** Severe struggle with sentence structure and syntax; limited control over simple patterns.
* **Score 2:** Limited understanding of syntax; frequent basic mistakes; incomplete sentences.
* **Score 3:** Decent grasp of sentence structure with grammatical errors, or decent grammar with syntax errors.
* **Score 4:** Strong understanding of syntax and grammar; minor, occasional errors; effective self-correction.
* **Score 5:** High grammatical accuracy; adept control of complex language structures; fluent self-repair.

### 1.2 Architectural Strategy: The Multimodal Dilemma
A key research question is whether to train directly on raw audio, use Speech-to-Text (ASR) transcripts, or apply an LLM-as-a-judge:
1. **Speech-to-Text (Transcripts):** Grammatical competence (clause structure, tense consistency, agreement, subordination) is fundamentally linguistic. Transcribing speech into tokens allows extracting rich grammatical and syntactic indicators.
2. **The ASR Auto-Correction Trap:** Modern ASR architectures (such as OpenAI Whisper) have implicit language models that "smooth out" or auto-correct spoken errors, stutters, and grammatical slips unless configured specifically with `temperature=0.0`, `condition_on_previous_text=False`, and VAD filtering.
3. **The 0.0 MOS Anomaly & Acoustic Grounding:** In `train.csv`, **37 samples have ground truth label `0.0`** (corresponding to non-speech, extreme noise, silence, or unresponsive candidates). Discarding audio completely would cause catastrophic error on these samples. We extract **physical acoustic energy (RMS)**, dynamic range, and silence ratios to ground the linguistic model.
4. **Why Not Raw LLM Judge Alone:** While an LLM judge provides qualitative rubric reasoning, raw LLM outputs suffer from discrete integer rounding and central tendency bias, leading to suboptimal Pearson Correlation and RMSE. Instead, we use a **Calibrated Multimodal Ensemble Regressor (Regularized Ridge + LightGBM)** that mathematically minimizes RMSE and maximizes Pearson $r$.


In [ ]:
# Core scientific and ML imports
import os
import sys
sys.path.append(".")
import re
import math
import json
import wave
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, Ridge
from lightgbm import LGBMRegressor

# Visual styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print("Environment initialized successfully.")


## 2. Exploratory Data Analysis & Target Distribution Diagnostics

We analyze the training dataset (`769` samples) and examine target properties.
Notice the distribution of MOS labels:
* Rubric scores span 1.0 to 5.0 with half-point increments.
* Ground truth contains a distinct cluster of 37 samples with score `0.0`.


In [ ]:
# Load train metadata
TRAIN_CSV = r"shl-hiring-assessment-2026/Dataset_Final/train.csv"
train_df = pd.read_csv(TRAIN_CSV)

print(f"Total Training Samples: {len(train_df)}")
print("\nTarget Label Statistics:")
print(train_df["label"].describe())

print("\nFrequency Distribution of MOS Labels:")
print(train_df["label"].value_counts().sort_index())

# Plot Target Distribution
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.countplot(data=train_df, x="label", palette="Blues_r", ax=ax)
ax.set_title("Distribution of Spoken Grammar MOS Scores (Train Set)", fontsize=13, fontweight="bold")
ax.set_xlabel("MOS Grammar Score (0.0 - 5.0)")
ax.set_ylabel("Sample Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height() + 2), ha='center', fontsize=10)
plt.tight_layout()
plt.show()


## 3. Acoustic Signal & Physical Energy Preprocessing

To capture spoken rhythm, silence pauses, and detect the `0.0` non-speech instances:
* **RMS Amplitude & Peak Energy:** Quantifies overall voice projection and signal strength.
* **Silence Ratio & Speech Duration:** Measures proportion of 25ms frames falling below -35dB from peak.
* **Energy Variance & Dynamic Range:** Differentiates continuous natural speech from background noise.
* **Non-speech / Corrupt Audio Flag:** Accurately isolates dead audio files.


In [ ]:
# Load precomputed acoustic features (or compute directly)
AUDIO_FEAT_PATH = "data/train_audio_features.csv"
if os.path.exists(AUDIO_FEAT_PATH):
    audio_feat_df = pd.read_csv(AUDIO_FEAT_PATH)
    print(f"Loaded Acoustic Features: {audio_feat_df.shape}")
    display(audio_feat_df.head(5))
    
    # Check correlation between acoustic features and grammar MOS
    corrs = audio_feat_df.select_dtypes(include=np.number).corr()["label"].sort_values(ascending=False)
    print("\nCorrelation of Acoustic Signals with Target MOS:")
    print(corrs)


## 4. Verbatim Transcription & Rubric-Aligned Linguistic Features

We transcribe spoken audio using `faster-whisper` (`tiny.en`, CPU int8 quantized) with strict verbatim parameters:
* `condition_on_previous_text=False` (prevents hallucinating repetitive text)
* `temperature=0.0` (deterministic verbatim output)
* `vad_filter=True` (removes non-speech intervals)

From the resulting transcripts, we extract **21 domain-specific features**:
1. **Syntactic Complexity:** Subordination conjunction density (`because`, `although`, `since`, `whereas`), relative clauses (`which`, `that`, `who`), complex modal auxiliary usage (`would have`, `could have`, `might`).
2. **Grammar & Sentence Completeness:** Mean sentence length, variance in sentence length, incomplete sentence ratio (dangling conjunctions/prepositions).
3. **Fluency & Disfluencies:** Spoken filler counts (`um`, `uh`, `like`), consecutive word repetition rate, self-correction repair phrases (`I mean`, `rather`, `sorry`).
4. **Lexical Sophistication:** Type-Token Ratio (TTR), Guiraud's Index ($V / \sqrt{N}$), polysyllabic word density.
5. **Speech Rate Metrics:** Speaking rate and words per minute (WPM).


In [ ]:
# Load transcripts and linguistic features
from src.linguistic_features import extract_dataset_linguistic_features, extract_single_linguistic_features

TRAIN_TRANS_PATH = "data/train_transcripts.csv"
if os.path.exists(TRAIN_TRANS_PATH):
    train_trans_df = pd.read_csv(TRAIN_TRANS_PATH)
    print(f"Loaded Transcripts: {train_trans_df.shape}")
    print("\nSample Transcripts across Score Bands:")
    for score in [2.0, 3.5, 5.0]:
        sample = train_trans_df[train_trans_df["label"] == score].head(1)
        if not sample.empty:
            print(f"\n--- MOS {score} Sample ({sample['filename'].values[0]}) ---")
            print(sample["transcript"].values[0][:250] + "...")


## 5. J-Eval / G-Eval LLM-as-a-Judge Rubric Engine

To incorporate qualitative rubric evaluations directly against the official human rubric:
* We implement **J-Eval (LLM-as-a-Judge)** using Google Gemini (`gemini-3.8-flash` / `gemini-flash-lite-latest`) to evaluate spoken transcripts across 4 multi-dimensional criteria:
  1. `syntax_score`: Subordination, relative clauses, clause depth, sentence complexity (1.0 - 5.0).
  2. `grammar_accuracy_score`: Morphosyntax, agreement, tense consistency, error density (1.0 - 5.0).
  3. `sentence_completion_score`: Incomplete sentences, dangling conjunctions (1.0 - 5.0).
  4. `judge_rubric_mos`: Overall continuous rubric verdict calibrated against the official rubric guidelines.
* **Why calibrate with Regressor:** The Judge's qualitative verdict provides a strong semantic anchor, which is calibrated alongside physical acoustics and statistical syntax features within the cross-validated ensemble regressor.


In [ ]:
# Live J-Eval Gemini LLM Judge integration
import os
import sys
sys.path.append(".")
from dotenv import load_dotenv
load_dotenv(".env")

# Embed the Official Rubric Prompt directly for 100% standalone reproducibility
RUBRIC_PROMPT_TEMPLATE = """You are the Lead Linguistic Assessor for the SHL Spoken Grammar competition.
Evaluate the spoken English grammar of this transcript based on the official rubric (Scores 1.0 to 5.0):

[TRANSCRIPT]
\"\"\"{transcript}\"\"\"

[OFFICIAL RUBRIC GUIDELINES]
Score 1: Speech struggles with proper sentence structure and syntax, displaying limited control over simple grammatical structures and memorized patterns.
Score 2: Limited understanding of sentence structure and syntax. Although they use simple structures, they consistently make basic grammatical mistakes and may leave sentences incomplete.
Score 3: Decent grasp of sentence structure but makes errors in grammatical structure, or decent grasp of grammar but makes errors in sentence syntax and structure.
Score 4: Strong understanding of sentence structure and syntax. Consistently good control of grammar; errors are minor and infrequent; person can self-correct most of them.
Score 5: High grammatical accuracy and adept control of complex grammar. Uses grammar accurately and effectively, seldom making noticeable mistakes; handles complex structures well.

Return a valid JSON object adhering strictly to this schema:
{{
  "syntax_score": float (1.0 to 5.0),
  "grammar_accuracy_score": float (1.0 to 5.0),
  "sentence_completion_score": float (1.0 to 5.0),
  "judge_rubric_mos": float (1.0 to 5.0 continuous score),
  "reasoning": string (concise linguistic rationale explaining the score)
}}"""

print("=" * 60)
print("OFFICIAL RUBRIC SYSTEM PROMPT FOR GEMINI JUDGE:")
print("=" * 60)
print(RUBRIC_PROMPT_TEMPLATE.replace("{transcript}", "[AUDIO_TRANSCRIPT_HERE]"))

# Load precomputed Judge verdicts
train_judge_df = pd.read_csv("data/train_judge_scores.csv")
test_judge_df = pd.read_csv("data/test_judge_scores.csv")

print(f"\nLoaded J-Eval Judge Verdicts: Train={train_judge_df.shape}, Test={test_judge_df.shape}")
print("\nSample Live Gemini Judge Verdicts on Test Samples with Linguistic Rationale:")
for idx, r in test_judge_df.head(4).iterrows():
    print(f"\n[{r['filename']}] Verdict MOS: {r['judge_rubric_mos']} | Syntax: {r['syntax_score']} | Grammar: {r['grammar_accuracy_score']}")
    print(f"Reasoning: {r['reasoning']}")


## 6. 5-Fold Stratified Cross-Validation & Regressor Architecture

### 6.1 Validation Scheme
To eliminate data leakage while honoring the continuous nature of the target:
* We construct **5 Stratified Folds** binned by target MOS intervals (`[0, 1.5]`, `2.0`, `2.5`, `3.0`, `3.5`, `4.0`, `4.5`, `5.0`).
* Each fold maintains identical class proportions.

### 6.2 Complementary Regressor Models
1. **Model A (Regularized Ridge Regressor):** Standardizes all features and fits an L2-regularized linear model with cross-validated alpha search (`RidgeCV`). Provides smooth continuous predictions and prevents overfitting on high-dimensional text statistics.
2. **Model B (LightGBM Regressor):** Captures non-linear feature interactions between acoustic energy, fluency rates, syntactic complexity, and the J-Eval Judge rubric verdicts.
3. **Weighted Ensemble Blend:** Optimizes prediction weights on Out-of-Fold (OOF) validation to strictly minimize RMSE and maximize Pearson correlation.


In [ ]:
from src.model_trainer import train_evaluate_pipeline

# Load merged feature sets: Acoustic + Linguistic + J-Eval LLM Judge
train_audio_df = pd.read_csv("data/train_audio_features.csv")
test_audio_df = pd.read_csv("data/test_audio_features.csv")
train_ling_df = pd.read_csv("data/train_linguistic_features.csv")
test_ling_df = pd.read_csv("data/test_linguistic_features.csv")
train_judge_df = pd.read_csv("data/train_judge_scores.csv")
test_judge_df = pd.read_csv("data/test_judge_scores.csv")

# Merge modalities
ling_cols = [c for c in train_ling_df.columns if c not in ["label", "duration_sec", "speech_duration_sec"]]
judge_cols = ["filename", "judge_rubric_mos", "grammar_accuracy_score", "syntax_score", "sentence_completion_score"]

train_full = train_audio_df.merge(train_ling_df[ling_cols], on="filename", how="inner")
train_full = train_full.merge(train_judge_df[judge_cols], on="filename", how="inner")

test_full = test_audio_df.merge(test_ling_df[ling_cols], on="filename", how="left")
test_full = test_full.merge(test_judge_df[judge_cols], on="filename", how="left")
for col in judge_cols[1:]:
    test_full[col] = test_full[col].fillna(3.0)

print(f"Combined Training Feature Matrix (Multimodal + J-Eval Judge): {train_full.shape}")
print(f"Combined Test Feature Matrix     (Multimodal + J-Eval Judge): {test_full.shape}")
assert len(test_full) == 216, f"Expected 216 test rows, got {len(test_full)}"

# Execute 5-Fold Cross-Validation
submission_df, results = train_evaluate_pipeline(train_full, test_full, n_splits=5, random_state=42)
submission_df.to_csv("submission.csv", index=False)
submission_df.to_csv("data/submission.csv", index=False)
print(f"Saved submission.csv with exactly {len(submission_df)} rows!")


## 7. Model Evaluation & Benchmark Results

### 6.1 Compulsory Requirement Display
> **MANDATORY ASSESSMENT METRIC:**  
> The training data RMSE is explicitly computed on all 769 training samples below.


In [ ]:
# Prominently display the compulsory Training Data RMSE alongside OOF metrics
train_m = results["train_metrics"]
oof_m = results["oof_metrics"]

metrics_df = pd.DataFrame([
    {
        "Split": "TRAINING DATA (Compulsory)",
        "Pearson Correlation (r)": f"{train_m['pearson_r']:.4f}",
        "RMSE": f"{train_m['rmse']:.4f}",
        "Spearman (rho)": f"{train_m['spearman_rho']:.4f}",
        "MAE": f"{train_m['mae']:.4f}",
    },
    {
        "Split": "OUT-OF-FOLD (OOF) VALIDATION",
        "Pearson Correlation (r)": f"{oof_m['pearson_r']:.4f}",
        "RMSE": f"{oof_m['rmse']:.4f}",
        "Spearman (rho)": f"{oof_m['spearman_rho']:.4f}",
        "MAE": f"{oof_m['mae']:.4f}",
    }
])

display(metrics_df)

print(f"\n{'*'*60}")
print(f"COMPULSORY TRAINING DATA RMSE: {train_m['rmse']:.4f}")
print(f"OUT-OF-FOLD (OOF) VALIDATION RMSE: {oof_m['rmse']:.4f}")
print(f"OUT-OF-FOLD (OOF) PEARSON CORRELATION (r): {oof_m['pearson_r']:.4f}")
print(f"{'*'*60}")


## 7. Model Interpretability & Visualizations

To thoroughly evaluate model behavior:
1. **Actual vs. Predicted MOS Scatter Plot:** Shows agreement along the diagonal with Pearson correlation trendline.
2. **Residual Distribution ($y - \hat{y}$):** Verifies zero-centered, normally distributed errors without systemic bias.
3. **Feature Importance Ranking:** Identifies the primary physical and linguistic indicators driving spoken grammar scores.


In [ ]:
# 1. Actual vs Predicted Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

y_true = np.array(results["y_true"])
y_oof = np.array(results["oof_predictions"])

axes[0].scatter(y_true, y_oof, alpha=0.5, color="#1f77b4", edgecolors="none")
axes[0].plot([0, 5], [0, 5], "r--", label="Ideal Identity (y = x)")
m, b = np.polyfit(y_true, y_oof, 1)
axes[0].plot(np.linspace(0, 5, 100), m * np.linspace(0, 5, 100) + b, "g-", label=f"Fit (r = {oof_m['pearson_r']:.3f})")
axes[0].set_title("Actual vs. Predicted Grammar MOS (OOF)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Ground Truth MOS")
axes[0].set_ylabel("Predicted MOS")
axes[0].set_xlim(-0.2, 5.2)
axes[0].set_ylim(-0.2, 5.2)
axes[0].legend()

# 2. Residual Distribution Plot
residuals = y_true - y_oof
sns.histplot(residuals, kde=True, ax=axes[1], color="#2ca02c", bins=25)
axes[1].axvline(0, color="r", linestyle="--")
axes[1].set_title(f"Residual Error Distribution (RMSE = {oof_m['rmse']:.3f})", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Residual (Actual - Predicted)")
axes[1].set_ylabel("Density")

plt.tight_layout()
plt.show()

# 3. Top Feature Importances (LightGBM)
feature_names = results["feature_names"]
importances = results["lgb_importances"]
feat_imp_df = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values("Importance", ascending=False).head(15)

plt.figure(figsize=(10, 5.5))
sns.barplot(data=feat_imp_df, y="Feature", x="Importance", palette="viridis")
plt.title("Top 15 Most Informative Features (LightGBM Importance)", fontsize=13, fontweight="bold")
plt.xlabel("Importance Score")
plt.ylabel("Engineered Feature")
plt.tight_layout()
plt.show()


## 8. Test Set Inference & Submission Integrity Verification

We verify that the generated submission file conforms strictly to the competition rules:
* Schema contains exactly `filename,label`.
* Number of rows matches test set (`216` samples).
* Predictions are continuous and strictly bounded within $[0.0, 5.0]$.
* No `NaN`, null, or infinity values.


In [ ]:
# Inspect final submission dataframe
print(f"Total Test Predictions: {len(submission_df)}")
print(f"Any missing values: {submission_df.isnull().sum().to_dict()}")
print(f"Prediction range: Min = {submission_df['label'].min():.4f}, Max = {submission_df['label'].max():.4f}")

# Display first 15 predictions
display(submission_df.head(15))

# Save confirmed submission
submission_df.to_csv("submission.csv", index=False)
print("Confirmed submission written to: submission.csv")
